In [75]:
import os, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
from scipy.stats import norm as scipy_norm

warnings.filterwarnings("ignore")

DATA_DIR  = r"."
OUT_DIR  = DATA_DIR


WQ_FILES = {
    "P1":   "P1_cleaned.xlsx",
    "P2":     "P2_cleaned.xlsx",
    "P3":  "P3_cleaned.xlsx",
    "P4":      "P4_cleaned.xlsx",
    "P5": "P5_cleaned.xlsx",
    "P6":  "P6_cleaned.xlsx",
}

NODE_ORDER  = ["P1", "P2", "P3", "P4", "P5", "P6"]
HYDRO_FILE  = os.path.join(DATA_DIR, "origin3_filled.xlsx")
METEO_FILE  = os.path.join(DATA_DIR, "jiangyu.xlsx")

WQ_COLS    = ["COD", "NH3N", "TP", "DO"]
HYDRO_COLS = ["T_water", "v", "Q"]
METEO_COLS = ["rain", "T_air"]
HYDRO_DIM  = 5


TRAIN_END     = pd.Timestamp("2024-11-09 20:00:00")
EARLYSTOP_END = pd.Timestamp("2024-11-20 20:00:00")
TEST_START    = pd.Timestamp("2024-11-21 00:00:00")
TEST_END      = pd.Timestamp("2024-12-31 20:00:00")

L_SR   = 8
STRIDE = 4


BATCH_SIZE   = 16
EPOCHS       = 150
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 30
LAMBDA_REG   = 0.1

HYDRO_HIDDEN = 64
GAT_HIDDEN   = 64
LSTM_HIDDEN  = 64
TF_HIDDEN    = 64
MEM_DIM      = 128
LATENT_DIM   = 32
DISC_HIDDEN  = 64
TIME_ENC_DIM = 4
N_HEADS      = 4
N_NODES = len(NODE_ORDER) 
N_WQ    = len(WQ_COLS)
N_HYDRO = 5
SEED_LIST = [42,0,10,20,30,40,50,60,70,80]
DETERMINISTIC = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed, deterministic=DETERMINISTIC):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark     = not deterministic

print(f"Device: {DEVICE}")
print(f"Seeds: {SEED_LIST}")

Device: cuda
Seeds: [42, 0, 10, 20, 30, 40, 50, 60, 70, 80]


In [76]:
# df_all = df_hydro.join(df_meteo, how='inner')
# df_all['month'] = df_all.index.month

# print("month | sample num | Q average | Q Max | rain average | rain>5 times | rain>0.5 times")
# print("-" * 90)
# for m in range(1, 13):
#     sub = df_all[df_all['month'] == m]
#     n = len(sub)
#     q_mean = sub['Q'].mean()
#     q_max = sub['Q'].max()
#     r_mean = sub['rain'].mean()
#     r_gt5 = (sub['rain'] > 5).sum()
#     r_gt05 = (sub['rain'] > 8).sum()
#     print(f"{m:>4} | {n:>6} | {q_mean:>7.3f} | {q_max:>8.3f} | {r_mean:>8.4f} | {r_gt5:>10} | {r_gt05:>12}")

In [77]:
DISTANCES = [
    2484.82,
     268.59,
    8182.55,
    1730.00,
    2090.58,
]
n_nodes = N_NODES

edge_index_tensor = torch.tensor(
    [list(range(N_NODES - 1)), list(range(1, N_NODES))],
    dtype=torch.long,
).to(DEVICE)

inv_dist = torch.tensor([1.0 / d for d in DISTANCES], dtype=torch.float32)
edge_weight_base_tensor = (inv_dist / inv_dist.max()).to(DEVICE)

print("edge_index_tensor:\n", edge_index_tensor)
print("edge_weight_base_tensor:", edge_weight_base_tensor)

edge_index_tensor:
 tensor([[0, 1, 2, 3, 4],
        [1, 2, 3, 4, 5]], device='cuda:0')
edge_weight_base_tensor: tensor([0.1081, 1.0000, 0.0328, 0.1553, 0.1285], device='cuda:0')


In [78]:
wq_dfs = {}
for station in NODE_ORDER:
    path = os.path.join(DATA_DIR, WQ_FILES[station])
    df   = pd.read_excel(path)

    df["datetime"] = pd.to_datetime(
        df["DATA"].astype(str).str.strip() + " " +
        df["TIME"].astype(str).str.strip()
    )
    df = df.set_index("datetime").sort_index()
    wq_dfs[station] = df[WQ_COLS]

timestamps_4h = pd.DatetimeIndex(wq_dfs[NODE_ORDER[0]].index)

wq_array = np.stack(
    [wq_dfs[s].values for s in NODE_ORDER], axis=1
).astype(np.float32)


obs_mask_array = (~np.isnan(wq_array)).astype(np.float32)

print(f"4h timestamps : {len(timestamps_4h)}  [{timestamps_4h[0]} → {timestamps_4h[-1]}]")
print(f"index dtype   : {timestamps_4h.dtype}")
print(f"wq_array      : {wq_array.shape}  (NaN where missing)")
print(f"obs_mask      : {obs_mask_array.shape}")
print(f"Overall missing rate : {(1 - obs_mask_array).mean():.1%}")


4h timestamps : 2196  [2024-01-01 00:00:00 → 2024-12-31 20:00:00]
index dtype   : datetime64[us]
wq_array      : (2196, 6, 4)  (NaN where missing)
obs_mask      : (2196, 6, 4)
Overall missing rate : 8.4%


In [ ]:
df_hydro = pd.read_excel(HYDRO_FILE, parse_dates=[0])
df_hydro.columns = ["datetime"] + HYDRO_COLS
df_hydro = df_hydro.set_index("datetime").sort_index()

df_meteo = pd.read_excel(METEO_FILE, parse_dates=[0])
df_meteo.columns = ["datetime"] + METEO_COLS
df_meteo = df_meteo.set_index("datetime").sort_index()

df_aux        = df_hydro.join(df_meteo, how="inner")
timestamps_1h = df_aux.index
aux_array     = df_aux.values.astype(np.float32)

print(f"1h timestamps : {len(timestamps_1h)}  [{timestamps_1h[0]} → {timestamps_1h[-1]}]")
print(f"aux_array     : {aux_array.shape}")
assert len(aux_array) >= len(wq_array) * 4
print("Length check passed.")

In [ ]:
df_hydro = pd.read_excel(HYDRO_FILE, parse_dates=[0])
df_meteo = pd.read_excel(METEO_FILE, parse_dates=[0])

print("HYDRO shape:", df_hydro.shape)
print("HYDRO cols :", df_hydro.columns.tolist())

print("METEO shape:", df_meteo.shape)
print("METEO cols :", df_meteo.columns.tolist())

In [80]:

idx_train_end     = np.searchsorted(timestamps_4h, TRAIN_END,     side="right")
idx_earlystop_end = np.searchsorted(timestamps_4h, EARLYSTOP_END, side="right")
idx_test_start    = np.searchsorted(timestamps_4h, TEST_START,    side="left")
idx_test_end      = np.searchsorted(timestamps_4h, TEST_END,      side="right")

print(f"Train      : 0 – {idx_train_end}")
print(f"Early-stop : {idx_train_end} – {idx_earlystop_end}")
print(f"Test       : {idx_test_start} – {idx_test_end}")

train_flat = wq_array[:idx_train_end].reshape(-1, len(WQ_COLS))
train_obs  = train_flat[~np.isnan(train_flat).any(axis=1)]
wq_scaler  = StandardScaler().fit(train_obs)

wq_filled = np.where(np.isnan(wq_array), 0.0, wq_array)
wq_norm   = wq_scaler.transform(
    wq_filled.reshape(-1, len(WQ_COLS))
).reshape(wq_array.shape).astype(np.float32)

wq_norm[obs_mask_array == 0] = np.nan

n_train_1h = idx_train_end * 4
aux_scaler = StandardScaler().fit(aux_array[:n_train_1h])
aux_norm   = aux_scaler.transform(aux_array).astype(np.float32)

print("Normalisation complete.")

ts1h_index = {ts: i for i, ts in enumerate(timestamps_1h)}

Train      : 0 – 1884
Early-stop : 1884 – 2010
Test       : 2010 – 2196
Normalisation complete.


In [81]:

train_flat = wq_array[:idx_train_end].reshape(-1, len(WQ_COLS))
train_obs  = train_flat[~np.isnan(train_flat).any(axis=1)]
wq_scaler  = StandardScaler().fit(train_obs)

wq_filled = np.where(np.isnan(wq_array), 0.0, wq_array)
wq_norm   = wq_scaler.transform(
    wq_filled.reshape(-1, len(WQ_COLS))
).reshape(wq_array.shape).astype(np.float32)

wq_norm[obs_mask_array == 0] = np.nan
n_train_1h = idx_train_end * 4
aux_scaler = StandardScaler().fit(aux_array[:n_train_1h])
aux_norm   = aux_scaler.transform(aux_array).astype(np.float32)


In [82]:
def make_time_enc(timestamps):
    hour = timestamps.hour.values
    doy  = timestamps.dayofyear.values
    return np.stack([
        np.sin(2 * np.pi * hour / 24),
        np.cos(2 * np.pi * hour / 24),
        np.sin(2 * np.pi * doy  / 366),
        np.cos(2 * np.pi * doy  / 366),
    ], axis=-1).astype(np.float32)

time_enc_4h = make_time_enc(timestamps_4h)
print(f"time_enc_4h shape: {time_enc_4h.shape}")

time_enc_4h shape: (2196, 4)


In [83]:
ts1h_index = {ts: i for i, ts in enumerate(timestamps_1h)}

class HGSTGNNDataset(Dataset):
    def __init__(self, start_idx, end_idx, mode="train", seed=42):
        self.starts = list(range(start_idx, end_idx - L_SR + 1, STRIDE))
        self.mode   = mode
        self.seed   = seed
        self.epoch  = 0

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s4 = self.starts[idx]
        wq_win_nan = wq_norm[s4 : s4 + L_SR].copy()
        gt_mask    = obs_mask_array[s4 : s4 + L_SR].copy()
        time_enc   = time_enc_4h[s4 : s4 + L_SR]
        s1         = ts1h_index[timestamps_4h[s4]]
        hydro      = aux_norm[s1 : s1 + L_SR * 4]

        wq_gt = np.where(np.isnan(wq_win_nan), 0.0, wq_win_nan).astype(np.float32)

        candidates = [t for t in range(1, L_SR - 1) if gt_mask[t].max() > 0]
        if not candidates:
            candidates = list(range(1, L_SR - 1))

        rng = np.random.default_rng(
            self.seed * 10000000 + self.epoch * 100000 + idx
        )
        mask_idx = int(rng.choice(candidates))

        wq_input = wq_gt.copy()
        obs_mask = gt_mask.copy()
        wq_input[mask_idx] = 0.0
        obs_mask[mask_idx] = 0.0

        eval_mask = np.zeros_like(gt_mask)
        eval_mask[mask_idx] = gt_mask[mask_idx]

        out = {
            "wq_input":  torch.from_numpy(wq_input),
            "obs_mask":  torch.from_numpy(obs_mask),
            "wq_gt":     torch.from_numpy(wq_gt),
            "gt_mask":   torch.from_numpy(gt_mask),
            "eval_mask": torch.from_numpy(eval_mask),
            "hydro":     torch.from_numpy(hydro),
            "time_enc":  torch.from_numpy(time_enc),
        }
        if self.mode == "train":
            out["mask_idx"] = torch.tensor(mask_idx, dtype=torch.long)
        return out


In [84]:
class VarCrossAttn(nn.Module):
    def __init__(self, n_wq=4, n_hydro=5, hydro_hidden=64):
        super().__init__()
        self.scale       = math.sqrt(hydro_hidden)
        self.wq_embed    = nn.Embedding(n_wq,    hydro_hidden)
        self.hydro_embed = nn.Embedding(n_hydro, hydro_hidden)
        self.val_proj    = nn.Linear(1, hydro_hidden)
        self.last_attn_weights = None

    def forward(self, hydro_4h):
        attn = torch.softmax(
            self.wq_embed.weight @ self.hydro_embed.weight.T / self.scale, dim=-1)
        V    = self.val_proj(hydro_4h.unsqueeze(-1))
        ctx  = torch.einsum("wh,blhd->blwd", attn, V)
        self.last_attn_weights = attn.detach()
        return ctx, attn


class HydroEncoder(nn.Module):
    def __init__(self, hydro_dim=5, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(hydro_dim, hidden_dim, num_layers=1, batch_first=True)

    def forward(self, hydro):
        B, T1h, _ = hydro.shape
        gru_out, _ = self.gru(hydro)
        h1h        = gru_out
        idx_4h     = torch.arange(3, T1h, 4, device=hydro.device)
        h4h        = gru_out[:, idx_4h, :]
        idx_start  = torch.arange(0, T1h, 4, device=hydro.device)
        Q          = hydro[:, idx_start, 2]
        return h1h, h4h, Q


class DynamicGATBranch(nn.Module):
    def __init__(self, in_dim, gat_hidden, edge_weight_base, edge_index, n_nodes=6):
        super().__init__()
        A0 = torch.zeros(n_nodes, n_nodes)
        for k in range(edge_index.shape[1]):
            s, d = int(edge_index[0, k]), int(edge_index[1, k])
            A0[d, s] = float(edge_weight_base[k])
        self.register_buffer("A0", A0)
        self.alpha = nn.Parameter(torch.tensor(0.1))
        self.W_m   = nn.Linear(in_dim, gat_hidden, bias=False)
        self.w_a   = nn.Linear(gat_hidden, 1, bias=False)
        self.W_z   = nn.Linear(in_dim + gat_hidden, gat_hidden)

    def forward(self, x, Q, edge_index=None):
        m     = self.W_m(x)
        beta  = torch.sigmoid(self.w_a(m))
        scale = (1 + self.alpha * Q).clamp(min=0)[:, :, None, None]
        a     = self.A0[None, None] * scale
        agg   = torch.einsum("blij,bljh->blih", a, beta * m)
        return F.relu(self.W_z(torch.cat([x, agg], dim=-1)))


class LSTMBranch(nn.Module):
    def __init__(self, in_dim, lstm_hidden, hydro_hidden):
        super().__init__()
        self.lstm_hidden = lstm_hidden
        self.lstm   = nn.LSTM(in_dim + hydro_hidden, lstm_hidden,
                              num_layers=1, batch_first=True)
        self.h0_net = nn.Linear(hydro_hidden, lstm_hidden)
        self.c0_net = nn.Linear(hydro_hidden, lstm_hidden)

    def forward(self, x, h4h):
        B, L, N, C = x.shape
        H  = self.lstm_hidden
        h0 = self.h0_net(h4h[:, 0]).unsqueeze(1).expand(B, N, H).reshape(1, B*N, H).contiguous()
        c0 = self.c0_net(h4h[:, 0]).unsqueeze(1).expand(B, N, H).reshape(1, B*N, H).contiguous()
        h4h_exp = h4h.unsqueeze(2).expand(B, L, N, -1).reshape(B*N, L, -1)
        x_flat  = x.permute(0, 2, 1, 3).reshape(B*N, L, C)
        out, _  = self.lstm(torch.cat([x_flat, h4h_exp], dim=-1), (h0, c0))
        return out.reshape(B, N, L, H).permute(0, 2, 1, 3)


class TransformerBranch(nn.Module):
    def __init__(self, in_dim, tf_hidden, n_heads=4, dropout=0.1, L=8, N=6):
        super().__init__()
        self.proj_in    = nn.Linear(in_dim, tf_hidden)
        self.pos_embed  = nn.Parameter(torch.randn(1, L, 1, tf_hidden) * 0.02)
        self.node_embed = nn.Parameter(torch.randn(1, 1, N, tf_hidden) * 0.02)
        self.attn  = nn.MultiheadAttention(tf_hidden, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(tf_hidden)
        self.ff    = nn.Sequential(nn.Linear(tf_hidden, tf_hidden * 2), nn.GELU(),
                                   nn.Linear(tf_hidden * 2, tf_hidden))
        self.norm2 = nn.LayerNorm(tf_hidden)

    def forward(self, x):
        B, L, N, C = x.shape
        z = self.proj_in(x) + self.pos_embed + self.node_embed
        z = z.reshape(B, L * N, -1)
        z2, _ = self.attn(z, z, z)
        z = self.norm1(z + z2)
        z = self.norm2(z + self.ff(z))
        return z.reshape(B, L, N, -1)


class ImputationHead(nn.Module):
    def __init__(self, mem_dim, n_wq=4):
        super().__init__()
        self.n_wq = n_wq
        self.fc = nn.Sequential(
            nn.Linear(mem_dim, mem_dim), nn.ReLU(),
            nn.Linear(mem_dim, n_wq * 2))

    def forward(self, memory):
        out   = self.fc(memory)
        mu    = out[..., :self.n_wq]
        sigma = F.softplus(out[..., self.n_wq:]) + 0.05
        return mu, sigma


class SRDecoder(nn.Module):
    def __init__(self, mem_dim, hydro_hidden, n_wq=4, n_nodes=6,
                 n_steps=28, n_heads=4, latent_dim=32, dropout=0.1):
        super().__init__()
        self.n_steps    = n_steps
        self.n_wq       = n_wq
        self.n_nodes    = n_nodes
        self.latent_dim = latent_dim
        self.mem_dim    = mem_dim

        self.pool_proj = nn.Linear(mem_dim, mem_dim)
        self.fc_mu_z   = nn.Linear(mem_dim, latent_dim)
        self.fc_lv_z   = nn.Linear(mem_dim, latent_dim)
        self.z_proj    = nn.Linear(latent_dim, mem_dim)
        self.pos_embed  = nn.Parameter(torch.randn(1, n_steps, mem_dim) * 0.02)

        self.node_embed = nn.Embedding(n_nodes, mem_dim)
        nn.init.normal_(self.node_embed.weight, std=0.05)
        self.hydro_proj = nn.Linear(hydro_hidden, mem_dim)

        self.dec_layer = nn.TransformerDecoderLayer(
            d_model=mem_dim, nhead=n_heads,
            dim_feedforward=mem_dim * 2,
            dropout=dropout, batch_first=True)
        self.dec_norm  = nn.LayerNorm(mem_dim)
        self.out_proj  = nn.Linear(mem_dim, n_wq * 2)

    def encode_latent(self, memory):
        B, L, N, D = memory.shape
        h = self.pool_proj(memory).mean(dim=(1, 2))
        return self.fc_mu_z(h), self.fc_lv_z(h)

    def reparameterise(self, mu_z, logvar_z):
        if self.training:
            return mu_z + torch.randn_like(mu_z) * torch.exp(0.5 * logvar_z)
        return mu_z

    def forward(self, memory, h1h, T1h_len):
        B, L, N, D = memory.shape

        mu_z, logvar_z = self.encode_latent(memory)
        z     = self.reparameterise(mu_z, logvar_z)
        z_mem = self.z_proj(z)

        step_hydro = []
        for i in range(L - 1):
            for j in range(1, 5):
                t_1h = min(i * 4 + j, T1h_len - 1)
                step_hydro.append(h1h[:, t_1h, :])
        hydro_ctx = self.hydro_proj(torch.stack(step_hydro, dim=1))

        node_emb = self.node_embed(torch.arange(N, device=memory.device))
        query = (self.pos_embed.unsqueeze(2).expand(B, -1, N, -1)
                 + hydro_ctx.unsqueeze(2).expand(B, -1, N, -1)
                 + z_mem.unsqueeze(1).unsqueeze(2)
                 + node_emb.unsqueeze(0).unsqueeze(0))
        query_flat = query.permute(0, 2, 1, 3).reshape(B * N, self.n_steps, D)

        kv = memory.reshape(B, L * N, D)
        kv = kv.unsqueeze(1).expand(-1, N, -1, -1).reshape(B * N, L * N, D)

        dec_out = self.dec_norm(self.dec_layer(query_flat, kv))
        out     = self.out_proj(dec_out)
        out     = out.reshape(B, N, self.n_steps, self.n_wq * 2).permute(0, 2, 1, 3)
        sr_mu    = out[..., :self.n_wq]
        sr_sigma = F.softplus(out[..., self.n_wq:]) + 0.05
        return sr_mu, sr_sigma, mu_z, logvar_z

class HGSTGNN(nn.Module):
    def __init__(
        self,
        n_nodes=6, n_wq=4, hydro_dim=5,
        hydro_hidden=64, gat_hidden=64, lstm_hidden=64, tf_hidden=64,
        mem_dim=128, time_enc_dim=4, n_heads=4, latent_dim=32,
        edge_weight_base=None, edge_index=None,
    ):
        super().__init__()
        self.register_buffer("edge_index", edge_index)
        self.mask_token = nn.Parameter(torch.randn(n_wq) * 0.5)

        self.hydro_encoder  = HydroEncoder(hydro_dim, hydro_hidden)
        self.var_cross_attn = VarCrossAttn(n_wq, hydro_dim, hydro_hidden)
        self.var_ctx_proj = nn.Linear(n_wq * hydro_hidden, hydro_hidden)

        in_dim_full = n_wq + n_wq + time_enc_dim + hydro_hidden
        self.input_proj      = nn.Linear(in_dim_full, gat_hidden)

        self.gat_branch  = DynamicGATBranch(gat_hidden, gat_hidden,
                                             edge_weight_base, edge_index, n_nodes)
        self.lstm_branch = LSTMBranch(gat_hidden, lstm_hidden, hydro_hidden)
        self.tf_branch   = TransformerBranch(gat_hidden, tf_hidden, n_heads)

        fused_dim = gat_hidden + lstm_hidden + tf_hidden
        self.fusion = nn.Sequential(nn.Linear(fused_dim, mem_dim), nn.ReLU())

        self.imputation_head = ImputationHead(mem_dim, n_wq)
        self.sr_decoder      = SRDecoder(
            mem_dim=mem_dim, hydro_hidden=hydro_hidden,
            n_wq=n_wq, n_nodes=n_nodes,
            n_steps=(8-1)*4, n_heads=n_heads, latent_dim=latent_dim)

    def encode(self, wq_input, obs_mask, hydro, time_enc):
        B, L, N, Q = wq_input.shape
        mask_bc = self.mask_token.view(1, 1, 1, Q).expand(B, L, N, Q)
        wq_enc  = torch.where(obs_mask.bool(), wq_input, mask_bc)

        h1h, h4h, Q_flow = self.hydro_encoder(hydro)

        idx_start  = torch.arange(0, hydro.shape[1], 4, device=hydro.device)
        hydro_4h   = hydro[:, idx_start, :]
        var_ctx, _   = self.var_cross_attn(hydro_4h)
        B_, L_ = var_ctx.shape[:2]
        var_ctx_flat = var_ctx.reshape(B_, L_, -1)
        var_ctx_comp = self.var_ctx_proj(var_ctx_flat)
        var_ctx_exp  = var_ctx_comp.unsqueeze(2).expand(B_, L_, N, -1)

        time_exp = time_enc.unsqueeze(2).expand(B, L, N, -1)
        x_in     = torch.cat([wq_enc, obs_mask, time_exp, var_ctx_exp], dim=-1)
        x_proj   = self.input_proj(x_in)

        gat_out  = self.gat_branch(x_proj, Q_flow, self.edge_index)
        lstm_out = self.lstm_branch(x_proj, h4h)
        tf_out   = self.tf_branch(x_proj)
        memory   = self.fusion(torch.cat([gat_out, lstm_out, tf_out], dim=-1))

        return memory, h1h, h4h, Q_flow

    def forward(self, wq_input, obs_mask, hydro, time_enc):
        memory, h1h, h4h, Q_flow = self.encode(wq_input, obs_mask, hydro, time_enc)
        imp_mu, imp_sigma = self.imputation_head(memory)
        sr_mu, sr_sigma, mu_z, logvar_z = self.sr_decoder(memory, h1h, h1h.shape[1])
        return imp_mu, imp_sigma, sr_mu, sr_sigma, mu_z, logvar_z

In [85]:
def variance_match_by_quantile(pred, target, mask, var_idx=None, n_bins=5):
    if var_idx is not None:
        pred   = pred[..., var_idx]
        target = target[..., var_idx]
        mask   = mask[..., var_idx]
    m = mask.bool()
    t = target[m]
    p = pred[m]
    if len(t) < n_bins * 5:
        return torch.tensor(0.0, device=pred.device)
    idx = torch.argsort(t)
    t_s, p_s = t[idx], p[idx]
    loss = 0.0
    for i in range(n_bins):
        s = i * len(t) // n_bins
        e = (i + 1) * len(t) // n_bins
        loss += (p_s[s:e].std() - t_s[s:e].std()) ** 2
    return loss / n_bins

def variance_match_weighted(imp_mu, wq_gt, obs_mask, val_ratios, n_bins=5):

    total = torch.tensor(0.0, device=imp_mu.device)
    for vi, var in enumerate(WQ_COLS):
        r = val_ratios.get(var, 1.0)
        w = max(0.0, r - 1.0)
        if w <= 0:
            continue
        loss_v = variance_match_by_quantile(
            imp_mu, wq_gt, obs_mask, var_idx=[vi], n_bins=n_bins)
        total = total + w * loss_v
    return total

In [86]:
LAMBDA_SR_ANCHOR = 0.5
LAMBDA_KL        = 5e-3 
W_OBS_MSE        = 1.0
W_OBS_NLL        = 0.1 
W_SELF_MSE       = 1.0
W_SELF_NLL       = 0.1
W_SELF_TOTAL     = 2.0


def gaussian_nll(pred_mu, pred_sigma, target, mask):
    denom = mask.sum()
    if denom == 0:
        return torch.tensor(0.0, device=pred_mu.device)
    var = pred_sigma ** 2 + 1e-2
    nll = 0.5 * (torch.log(var) + (pred_mu - target) ** 2 / var)
    return (nll * mask).sum() / denom


def mse_masked(pred, target, mask):
    denom = mask.sum()
    if denom == 0:
        return torch.tensor(0.0, device=pred.device)
    return ((pred - target) ** 2 * mask).sum() / denom


def compute_generator_loss(
    imp_mu, imp_sigma, sr_mu, sr_sigma,
    obs_mask, gt_mask, wq_gt, mask_idx,
    mu_z=None, logvar_z=None):

    B = imp_mu.shape[0]

    loss_obs = (W_OBS_MSE * mse_masked(imp_mu, wq_gt, obs_mask)
                + W_OBS_NLL * gaussian_nll(imp_mu, imp_sigma, wq_gt, obs_mask))


    loss_self = torch.tensor(0.0, device=imp_mu.device)
    for b in range(B):
        m = mask_idx[b].item()
        node_gt = gt_mask[b, m, :, :]
        if node_gt.sum() == 0:
            continue
        loss_self += (W_SELF_MSE * mse_masked(
                          imp_mu[b, m:m+1], wq_gt[b, m:m+1], node_gt.unsqueeze(0))
                      + W_SELF_NLL * gaussian_nll(
                          imp_mu[b, m:m+1], imp_sigma[b, m:m+1],
                          wq_gt[b, m:m+1], node_gt.unsqueeze(0)))
    loss_self = W_SELF_TOTAL * loss_self / B
    sr_4h      = sr_mu[:, 3::4]
    imp_next   = imp_mu[:, 1:]
    next_mask  = gt_mask[:, 1:].float()
    loss_sr_anchor = mse_masked(sr_4h, imp_next, next_mask)

    loss_kl = torch.tensor(0.0, device=imp_mu.device)
    if mu_z is not None and logvar_z is not None:
        loss_kl = -0.5 * torch.mean(1 + logvar_z - mu_z.pow(2) - logvar_z.exp())


    loss_var = variance_match_weighted(
        imp_mu, wq_gt, obs_mask, val_ratios, n_bins=5)

    total = (loss_obs
             + loss_self
             + LAMBDA_SR_ANCHOR * loss_sr_anchor
             + LAMBDA_KL * loss_kl
             + 0.1 * loss_var)

    return total, loss_obs.item(), loss_self.item(), loss_sr_anchor.item(), loss_kl.item(), loss_var.item()

In [87]:
def inverse_wq(arr_norm):
    shape = arr_norm.shape
    return wq_scaler.inverse_transform(
        arr_norm.reshape(-1, len(WQ_COLS))
    ).reshape(shape).astype(np.float32)

def compute_metrics(pred, true):
    mae  = np.mean(np.abs(pred - true))
    rmse = np.sqrt(np.mean((pred - true) ** 2))
    denom = np.abs(true)
    valid = denom > 1e-6
    mape  = np.mean(np.abs((pred[valid] - true[valid]) / denom[valid])) * 100 \
            if valid.any() else np.nan
    ss_res = np.sum((true - pred) ** 2)
    ss_tot = np.sum((true - true.mean()) ** 2)
    nse    = 1 - ss_res / (ss_tot + 1e-10)
    r      = np.corrcoef(true, pred)[0, 1] if true.std() > 1e-10 else 0.0
    alpha  = pred.std() / (true.std() + 1e-10)
    beta   = pred.mean() / (true.mean() + 1e-10)
    kge    = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    sigma  = np.std(pred - true) + 1e-10
    z      = (true - pred) / sigma
    crps   = (sigma * (z * (2 * scipy_norm.cdf(z) - 1)
              + 2 * scipy_norm.pdf(z) - 1 / np.sqrt(np.pi))).mean()
    cov90  = np.mean((true >= pred - 1.645*sigma) & (true <= pred + 1.645*sigma)) * 100
    cov95  = np.mean((true >= pred - 1.960*sigma) & (true <= pred + 1.960*sigma)) * 100
    cov99  = np.mean((true >= pred - 2.576*sigma) & (true <= pred + 2.576*sigma)) * 100
    return dict(MAE=mae, RMSE=rmse, MAPE=mape,
                NSE=nse, KGE=kge, CRPS=crps,
                Cov90=cov90, Cov95=cov95, Cov99=cov99)

In [88]:
val_ratios = {
    "COD":  1.6521,
    "NH3N": 3.2107,
    "TP":   1.6270,
    "DO":   1.0706,
}

print("val_ratios (frozen from baseline):")
for var, r in val_ratios.items():
    print(f"  {var}: ratio={r:.4f}  weight={max(0, r-1):.4f}")

val_ratios (frozen from baseline):
  COD: ratio=1.6521  weight=0.6521
  NH3N: ratio=3.2107  weight=2.2107
  TP: ratio=1.6270  weight=0.6270
  DO: ratio=1.0706  weight=0.0706


In [91]:
SEEDS = [42, 0, 10, 20, 30, 40, 50, 60, 70, 80]
TAG   = "42t_hgstgnn42"
NPY_PATTERN = "42t_hgstgnn_1h_seed{seed}.npy"

OUT_SUMMARY  = f"{TAG}_1h_metrics.xlsx"
OUT_PER_SEED = f"{TAG}_1h_metrics_per_seed.xlsx"

T_4h = len(timestamps_4h)
N    = len(NODE_ORDER)
V    = len(WQ_COLS)

results = {}

for seed in SEEDS:
    path = NPY_PATTERN.format(seed=seed)
    if not os.path.exists(path):
        print(f"[SEED={seed}] 跳过: {path} 不存在")
        continue
    print(f"\n[SEED={seed}] 读取 {path}")

    arr_1h = np.load(path)
    T_1h   = arr_1h.shape[0]

    per_seed = {}
    for si, station in enumerate(NODE_ORDER):
        for vi, var in enumerate(WQ_COLS):
            x = arr_1h[:, si, vi]

            x_anchor = x[::4]
            t_anchor = wq_array[:, si, vi]
            n_min    = min(len(x_anchor), len(t_anchor))
            x_a      = x_anchor[:n_min]
            t_a      = t_anchor[:n_min]
            m_valid  = ~np.isnan(t_a)

            if m_valid.sum() >= 5:
                diff = x_a[m_valid] - t_a[m_valid]
                anchor_mae  = float(np.abs(diff).mean())
                anchor_rmse = float(np.sqrt((diff ** 2).mean()))
                anchor_bias = float(diff.mean())
            else:
                anchor_mae = anchor_rmse = anchor_bias = np.nan

            n_windows = (T_1h - 1) // 4
            devs = []
            for w in range(n_windows):
                a_left  = x[w * 4]
                a_right = x[(w + 1) * 4] if (w + 1) * 4 < T_1h else x[-1]
                for j in [1, 2, 3]:
                    t = j / 4.0
                    interp = a_left * (1 - t) + a_right * t
                    idx = w * 4 + j
                    if idx < T_1h:
                        devs.append(abs(x[idx] - interp))
            dev_from_interp = float(np.mean(devs)) if devs else np.nan

            x_valid = x[~np.isnan(x)]
            if len(x_valid) >= 5:
                x_mean = float(x_valid.mean())
                x_std  = float(x_valid.std())
                xc = x_valid - x_valid.mean()
                denom = (xc ** 2).sum()
                ac1 = float((xc[:-1] * xc[1:]).sum() / denom) if denom > 1e-12 else np.nan
            else:
                x_mean = x_std = ac1 = np.nan

            per_seed[(station, var)] = {
                "anchor_MAE":       anchor_mae,
                "anchor_RMSE":      anchor_rmse,
                "anchor_bias":      anchor_bias,
                "dev_from_interp":  dev_from_interp,
                "series_mean":      x_mean,
                "series_std":       x_std,
                "series_ac1":       ac1,
            }

    results[seed] = per_seed
    print(f"  完成: {len(per_seed)} 个 (station, var)")


print("\n" + "=" * 100)
print("跨 seed 汇总: 各站 × 变量 的指标均值 ± std")
print("=" * 100)

METRIC_KEYS = ["anchor_MAE", "anchor_RMSE", "anchor_bias",
               "dev_from_interp", "series_mean", "series_std", "series_ac1"]

summary_rows = []
for station in NODE_ORDER:
    for var in WQ_COLS:
        key = (station, var)
        row = {"Station": station, "Var": var}
        for mk in METRIC_KEYS:
            vals = [results[s][key][mk] for s in SEEDS
                    if s in results and key in results[s]
                    and not np.isnan(results[s][key][mk])]
            row[f"{mk}_mean"] = np.mean(vals) if vals else np.nan
            row[f"{mk}_std"]  = np.std(vals)  if vals else np.nan
        summary_rows.append(row)
df = pd.DataFrame(summary_rows)

df.to_excel(OUT_SUMMARY, index=False)
print(f"[Saved] {OUT_SUMMARY}")

per_seed_rows = []
for s in SEEDS:
    if s not in results:
        continue
    for (station, var), m in results[s].items():
        per_seed_rows.append({
            "Seed": s, "Station": station, "Var": var, **m
        })
pd.DataFrame(per_seed_rows).to_excel(OUT_PER_SEED, index=False)
print(f"[Saved] {OUT_PER_SEED}")

for var in WQ_COLS:
    sub = df[df["Var"] == var]
    print(f"\n── {var} ──")
    print(f"  {'Station':<16} {'anchor_MAE':>16} {'dev_from_interp':>18} "
          f"{'series_std':>14} {'series_ac1':>14}")
    print("  " + "-" * 84)
    for _, r in sub.iterrows():
        print(f"  {r['Station']:<16} "
              f"{r['anchor_MAE_mean']:>8.4f}±{r['anchor_MAE_std']:<6.4f} "
              f"{r['dev_from_interp_mean']:>8.4f}±{r['dev_from_interp_std']:<8.4f} "
              f"{r['series_std_mean']:>7.4f} "
              f"{r['series_ac1_mean']:>7.4f}")

评估 B: 从 1h 序列算特征指标 (TAG=42t_hgstgnn42)

[SEED=42] 读取 42t_hgstgnn_1h_seed42.npy
  完成: 24 个 (station, var)

[SEED=0] 读取 42t_hgstgnn_1h_seed0.npy
  完成: 24 个 (station, var)

[SEED=10] 读取 42t_hgstgnn_1h_seed10.npy
  完成: 24 个 (station, var)

[SEED=20] 读取 42t_hgstgnn_1h_seed20.npy
  完成: 24 个 (station, var)

[SEED=30] 读取 42t_hgstgnn_1h_seed30.npy
  完成: 24 个 (station, var)

[SEED=40] 读取 42t_hgstgnn_1h_seed40.npy
  完成: 24 个 (station, var)

[SEED=50] 读取 42t_hgstgnn_1h_seed50.npy
  完成: 24 个 (station, var)

[SEED=60] 读取 42t_hgstgnn_1h_seed60.npy
  完成: 24 个 (station, var)

[SEED=70] 读取 42t_hgstgnn_1h_seed70.npy
  完成: 24 个 (station, var)

[SEED=80] 读取 42t_hgstgnn_1h_seed80.npy
  完成: 24 个 (station, var)

跨 seed 汇总: 各站 × 变量 的指标均值 ± std
[Saved] 42t_hgstgnn42_1h_metrics.xlsx
[Saved] 42t_hgstgnn42_1h_metrics_per_seed.xlsx

── COD ──
  Station                anchor_MAE    dev_from_interp     series_std     series_ac1
  ------------------------------------------------------------------------------------
  D